In [ ]:
!gdown 1WNlDBADwlaheMaVpIjEeAMklw7jle9Ja

In [ ]:
!unzip /kaggle/working/woodscape_input.zip
!rm /kaggle/working/woodscape_input.zip

In [ ]:
!gdown 13k17SjgQHZCO-1Ctr3DY_bW6DGvQZZie

In [ ]:
# --- Occlusion score без пространственного веса ---
ordinal_weights = torch.tensor([0.0, 1/3, 2/3, 1.0])  # веса под Clear/Transparent/Semi/Opaque

def compute_occlusion_score(logits):
    probs = torch.softmax(logits, dim=1)  # (1, 4, H, W)
    s_map = (probs * ordinal_weights.view(1, 4, 1, 1)).sum(dim=1)  # (1, H, W)
    score = s_map.mean().item()  # среднее по всем пикселям кадра
    return score

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

inputs = Path('/kaggle/working/woodscape_input')
gtLabels = inputs / 'gtLabels'
rgbImages = inputs / 'rgbImages'
rgbLabels = inputs / 'rgbLabels'

all_gt = sorted(gtLabels.iterdir())
all_rgb = sorted(rgbImages.iterdir())
all_rgblab = sorted(rgbLabels.iterdir())

step = 9
files_gt = all_gt[::step][:50]
files_rgb = all_rgb[::step][:50]
files_rgblab = all_rgblab[::step][:50]

n = len(files_gt)
fig, axes = plt.subplots(n, 3, figsize=(9, n * 3))

to_save = []


for i, (f1, f2, f3) in enumerate(zip(files_gt, files_rgb, files_rgblab)):
    img1 = Image.open(f1)
    img2 = Image.open(f2)
    img3 = Image.open(f3)
    to_save.append([f1, f2, f3])

    axes[i, 0].imshow(img1)
    axes[i, 0].set_title(f1.name, fontsize=7)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(img2)
    axes[i, 1].set_title(f2.name, fontsize=7)
    axes[i, 1].axis('off')

    axes[i, 2].imshow(img3)
    axes[i, 2].set_title(f3.name, fontsize=7)
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import os

keep_gt = {f1.name for f1, f2, f3 in to_save}
keep_rgb = {f2.name for f1, f2, f3 in to_save}
keep_rgblab = {f3.name for f1, f2, f3 in to_save}

folders_and_keep = [
    (gtLabels, keep_gt),
    (rgbImages, keep_rgb),
    (rgbLabels, keep_rgblab),
]

DRY_RUN = False  # поставь False, когда проверишь список и будешь уверен

for folder, keep_set in folders_and_keep:
    to_delete = [f for f in folder.iterdir() if f.name not in keep_set]
    print(f"{folder.name}: оставляем {len(keep_set)}, удаляем {len(to_delete)}")
    
    for f in to_delete:
        if DRY_RUN:
            pass 
        else:
            os.remove(f)

if DRY_RUN:
    print("\nЭто был dry-run, ничего не удалено. Поставь DRY_RUN = False для реального удаления.")

In [ ]:
!unzip /kaggle/working/model_outputs.zip
!rm /kaggle/working/model_outputs.zip

In [ ]:
!pip install segmentation_models_pytorch

In [ ]:
import torch
import segmentation_models_pytorch as smp
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import re
from IPython.display import display, clear_output

# --- Настройки ---
inputs = Path('/kaggle/working/woodscape_input')
gtLabels = inputs / 'gtLabels'
rgbImages = inputs / 'rgbImages'

model_outputs = Path('/kaggle/working/model_outputs')

model_folders = [
    'fpn_resnet18_focal_loss_all_files',
    'linknet_resnet18_torch_cross_entropy_all_files',
    'unet_resnet18_torch_cross_entropy_all_files',
]

arch_map = {
    'fpn': smp.FPN,
    'linknet': smp.Linknet,
    'unet': smp.Unet,
    'unetplusplus': smp.UnetPlusPlus,
    'manet': smp.MAnet,
    'pan': smp.PAN,
    'pspnet': smp.PSPNet,
    'deeplabv3plus': smp.DeepLabV3Plus,
    'deeplabv3': smp.DeepLabV3,
}

IMG_SIZE = (480, 640)  # поправь под реальный размер, если предсказания выглядят криво

def parse_model_name(folder_name):
    for arch in sorted(arch_map.keys(), key=len, reverse=True):
        if folder_name.startswith(arch + '_'):
            rest = folder_name[len(arch)+1:]
            encoder_match = re.match(r'(resnet\d+)', rest)
            encoder = encoder_match.group(1) if encoder_match else 'resnet18'
            return arch, encoder
    raise ValueError(f"Не смог распознать архитектуру в {folder_name}")

def load_model(folder_name):
    arch, encoder = parse_model_name(folder_name)
    model = arch_map[arch](encoder_name=encoder, encoder_weights=None, classes=4)

    ckpt_path = model_outputs / folder_name / 'model' / f"{arch}_{encoder}" / f"{arch}_{encoder}.ckpt"

    # weights_only=False — доверяем, т.к. чекпоинт свой, не из внешнего источника
    checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    state_dict = checkpoint.get('state_dict', checkpoint)

    new_state_dict = {}
    for k, v in state_dict.items():
        new_key = k[len('model.'):] if k.startswith('model.') else k
        new_state_dict[new_key] = v

    missing, unexpected = model.load_state_dict(new_state_dict, strict=False)
    if missing or unexpected:
        print(f"[{folder_name}] ⚠ missing={len(missing)}, unexpected={len(unexpected)} ключей")

    model.eval()
    return model

palette = {
    0: (34, 139, 34),
    1: (255, 215, 0),
    2: (255, 140, 0),
    3: (220, 20, 60),
}

def mask_to_rgb(mask):
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for cls, color in palette.items():
        rgb[mask == cls] = color
    return rgb

def preprocess(img_path):
    img = Image.open(img_path).convert('RGB').resize(IMG_SIZE[::-1])
    arr = np.array(img).astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    arr = (arr - mean) / std
    tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).float()
    return img, tensor

# --- Загружаем модели один раз (это по-прежнему быстро, не выводит картинок) ---
print("Загружаю модели...")
models = {name: load_model(name) for name in model_folders}
print("Готово, начинаю прогон по кадрам:\n")

# --- Файлы ---
all_rgb = sorted(rgbImages.iterdir())
already_seen = set(all_rgb[::9][:50])
remaining = [f for f in all_rgb if f not in already_seen]
files_rgb = remaining[:20]

# --- Отрисовка построчно, с немедленным показом ---
n_cols = 2 + len(models)

for i, rgb_path in enumerate(files_rgb):
    gt_path = gtLabels / rgb_path.name
    orig_img, tensor = preprocess(rgb_path)
    gt_mask = np.array(Image.open(gt_path).resize(IMG_SIZE[::-1], Image.NEAREST))

    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))

    axes[0].imshow(orig_img)
    axes[0].set_title(rgb_path.name, fontsize=8)
    axes[0].axis('off')

    axes[1].imshow(mask_to_rgb(gt_mask))
    axes[1].set_title('GT', fontsize=8)
    axes[1].axis('off')

    for j, (name, model) in enumerate(models.items()):
        with torch.no_grad():
            logits = model(tensor)
            pred = torch.argmax(logits, dim=1).squeeze(0).numpy()
        axes[2 + j].imshow(mask_to_rgb(pred))
        axes[2 + j].set_title(name.split('_')[0], fontsize=8)
        axes[2 + j].axis('off')

    plt.tight_layout()
    plt.show()  # ← показывает эту строку СРАЗУ, не дожидаясь остальных 19

In [ ]:
!gdown 1-VkbyXSP9a7VxBXCnfa3hpgwuFLmE_U8

In [ ]:
!unzip ./defocus_test_woodscape.zip

In [ ]:
for i, img_path in enumerate(defocus_files):
    orig_img, tensor = preprocess(img_path)

    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))

    axes[0].imshow(orig_img)
    axes[0].set_title(img_path.name, fontsize=8)
    axes[0].axis('off')

    for j, (name, model) in enumerate(models.items()):
        with torch.no_grad():
            logits = model(tensor)
            pred = torch.argmax(logits, dim=1).squeeze(0).numpy()
            score = compute_occlusion_score(logits)

        axes[1 + j].imshow(mask_to_rgb(pred))
        axes[1 + j].set_title(f"{name.split('_')[0]}\nS={score:.3f}", fontsize=8)
        axes[1 + j].axis('off')

    plt.tight_layout()
    plt.show()